# Práctica 2 · pix2pix: de máscara a cara
Generar una cara a partir de su máscara de segmentación (CelebAMask-HQ, 64×64).

In [ ]:
# Dependencias y configuración reproducible
import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
from tensorflow.keras import layers, models

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_DIR = Path("data")
IMG_DIR = DATA_DIR / "images"     # 0.jpg ... 29999.jpg
MASK_DIR = DATA_DIR / "masks"     # 00000_hair.png, 00000_nose.png, ...
IMG_SIZE = 64
N_IMAGENES = 30000                # bájalo (p. ej. 2000) para pruebas rápidas

# Partes de la cara que hay en data/masks (un canal por parte)
PARTES = ["hair", "l_eye", "r_eye", "nose", "mouth", "l_ear", "r_ear", "hat"]

## 1. Carga y visualización del dataset

In [ ]:
def cargar_imagen(i):
    img = Image.open(IMG_DIR / f"{i}.jpg").convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    return np.asarray(img, dtype=np.float32) / 255.0


def cargar_mascara(i):
    """Devuelve un array (64, 64, len(PARTES)) con un canal binario por parte.
    Si una parte no existe para esa cara, su canal queda a 0."""
    mascara = np.zeros((IMG_SIZE, IMG_SIZE, len(PARTES)), dtype=np.float32)
    for c, parte in enumerate(PARTES):
        ruta = MASK_DIR / f"{i:05d}_{parte}.png"
        if ruta.exists():
            m = Image.open(ruta).convert("L").resize((IMG_SIZE, IMG_SIZE), Image.NEAREST)
            mascara[..., c] = (np.asarray(m) > 127).astype(np.float32)
    return mascara


imagenes, mascaras = [], []
for i in range(N_IMAGENES):
    imagenes.append(cargar_imagen(i))
    mascaras.append(cargar_mascara(i))

print("Número de pares cargados:", len(imagenes))
print("Dimensiones imagen:", imagenes[0].shape, "| máscara:", mascaras[0].shape)

In [ ]:
# Colores para pintar la máscara (uno por parte)
COLORES = np.array([
    [0.55, 0.27, 0.07],  # hair
    [0.00, 0.60, 1.00],  # l_eye
    [0.00, 0.30, 1.00],  # r_eye
    [1.00, 0.80, 0.00],  # nose
    [0.90, 0.10, 0.20],  # mouth
    [0.20, 0.80, 0.20],  # l_ear
    [0.10, 0.50, 0.10],  # r_ear
    [0.60, 0.20, 0.80],  # hat
], dtype=np.float32)


def mascara_a_color(m):
    """Convierte la máscara (64, 64, n_partes) en una imagen RGB para visualizarla."""
    rgb = np.zeros(m.shape[:2] + (3,), dtype=np.float32)
    for c in range(m.shape[-1]):
        rgb[m[..., c] > 0] = COLORES[c]
    return rgb


# Mostramos algunos pares máscara → cara
plt.figure(figsize=(12, 5))
for k in range(5):
    n = random.randint(0, len(imagenes) - 1)
    plt.subplot(2, 5, k + 1)
    plt.imshow(mascara_a_color(mascaras[n]))
    plt.title(f"Máscara {n}")
    plt.axis("off")
    plt.subplot(2, 5, k + 6)
    plt.imshow(imagenes[n])
    plt.title("Cara real")
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Convertir a arrays 4D y separar entrenamiento / validación
X = np.array(mascaras, dtype=np.float32)   # entrada: máscara
Y = np.array(imagenes, dtype=np.float32)   # salida: cara
del imagenes, mascaras                     # libera RAM

indices = np.random.permutation(len(X))
n_val = int(0.1 * len(X))
val_idx, train_idx = indices[:n_val], indices[n_val:]

X_train, Y_train = X[train_idx], Y[train_idx]
X_val, Y_val = X[val_idx], Y[val_idx]
print("Entrenamiento:", X_train.shape, "→", Y_train.shape)
print("Validación:   ", X_val.shape, "→", Y_val.shape)

---

## 2. Arquitectura pix2pix

### GENERADOR (U-Net)

In [ ]:
# TODO: generador U-Net  máscara (64, 64, len(PARTES)) → cara (64, 64, 3)

### DISCRIMINADOR (PatchGAN)

In [ ]:
# TODO: discriminador PatchGAN que recibe (máscara, cara) y decide real / generada

### ENTRENAMIENTO

In [ ]:
# TODO: pérdidas (adversarial + L1), optimizadores y bucle de entrenamiento

In [ ]:
# TODO: curvas de pérdida del generador y del discriminador

### RESULTADOS

In [ ]:
# TODO: máscara de validación → cara generada → cara real

---

## 3. Conclusiones